**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Graph Signal Processing & GNNs

A flagship IEEE-SPS research area with almost no accessible teaching material: signals that live on **networks** — sensor grids, social graphs, molecules, power grids. Four sessions: the graph Laplacian gives graphs a Fourier transform, filters, and sampling theory — and message-passing GNNs drop out as learned graph filters. The oracle throughout: on a ring graph, everything must reduce to classical DSP.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3 (eigendecomposition — this course is its victory lap).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (classical Fourier, for the reduction check).
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) for Session 4.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def laplacian(A):
    return np.diag(A.sum(1)) - A

# our running graph: a random sensor network (geometric graph)
n_nodes = 80
pos = rng.random((n_nodes, 2))
D2 = ((pos[:, None] - pos[None]) ** 2).sum(-1)
A = ((D2 < 0.045) & (D2 > 0)).astype(float)
L = laplacian(A)
lam, U = np.linalg.eigh(L)                          # the graph's "frequencies" and "Fourier basis"

def draw(signal, title="", ax=None):
    if ax is None: fig, ax = plt.subplots(figsize=(3.6, 3.2))
    for i, j in zip(*np.nonzero(np.triu(A))):
        ax.plot(*zip(pos[i], pos[j]), "k-", linewidth=0.3, alpha=0.4)
    sc = ax.scatter(*pos.T, c=signal, s=45, cmap="coolwarm")
    ax.set_title(title, fontsize=9); ax.axis("off")
    return sc

---
### 🕐 Session 1 of 4 — *The Graph Laplacian & Graph Fourier Transform* (~40 min)
**Goal:** give any graph a frequency axis; verify it reduces to the DFT on a ring.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S3. &nbsp; **Feeds into:** Session 2 (filtering on graphs).

---

<details>
<summary>🎓 <b>Teacher notes — Session 1: The Graph Laplacian & Graph Fourier Transform</b></summary>

**Timing (~40 min).** 10 min "what is frequency without time?" · 12 min the Laplacian quadratic form · 8 min the harmonics · 10 min the ring oracle.

**Open with the question the session answers.** On a sensor network there is no time axis and no notion of "one sample later" — so what could *frequency* possibly mean? Let the room struggle for a moment. The answer is the reframe that carries the whole workshop: frequency is **smoothness with respect to the edges**. A low-frequency signal is one where connected nodes agree; a high-frequency one is where they disagree.

**Then make that precise with the quadratic form.** $x^\top L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ — literally the total disagreement across edges. Write it out and verify it from $L = D - A$ on a triangle; it takes two minutes and it converts the Laplacian from a definition into a meaning. Eigenvectors of $L$ ordered by eigenvalue are then the graph's own harmonics, and $\lambda$ *is* the frequency axis.

**The GFT is a change of basis, nothing more.** $\hat x = U^\top x$ is analysis in the eigenbasis — the [Hilbert space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) projection idea again, with the graph choosing the basis instead of a mathematician. Students who see it that way stop treating GSP as a new subject; it is the same machinery pointed at a different orthonormal basis.

**The ring oracle is the moment to protect — do not rush it.** On a ring graph, the Laplacian eigenvalues must be $2 - 2\cos(2\pi k/N)$, and the check comes back at 3.6e-15. That is not a sanity test, it is a *reduction*: classical DSP is the special case of GSP where the graph happens to be a ring. Ask the room what graph corresponds to an image — a 2-D grid — and what to a video. The generalisation is genuine, and this one line proves the new theory contains the old one rather than competing with it.

**Ask the room.** "Eigenvector 0 has $\lambda = 0$. What does it look like?" Constant — the DC component, because a constant signal has zero disagreement across every edge. Every connected graph has exactly one such eigenvector, and the multiplicity of $\lambda = 0$ counts connected components. That is a genuinely useful diagnostic: if your graph Fourier basis has three zero eigenvalues, your "network" is actually three networks.

**Point at eigenvector 60 on the plot.** The colours alternate sharply between neighbours — that is what high graph frequency looks like, and it is the visual analogue of $(-1)^n$ in classical DSP. Comparing eigenvector 0, 1, 4, and 60 across the panels builds the intuition faster than any formula.
</details>

## 2. Frequency Without Time

💡 **Intuition.** What does 'frequency' mean with no time axis? **Smoothness with respect to the edges.** The Laplacian quadratic form $x^T L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges — so Laplacian eigenvectors, ordered by eigenvalue, are the graph's own harmonics: $\lambda \approx 0$ ⇒ smooth (neighbors agree), large $\lambda$ ⇒ oscillatory (neighbors alternate). The **graph Fourier transform** is just analysis in this eigenbasis: $\hat{x} = U^T x$ — [Hilbert-space](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb) change of basis, with the graph choosing the basis.

In [2]:
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8))
for ax, k in zip(axes, [0, 1, 4, 60]):
    draw(U[:, k], f"eigenvector {k}: λ={lam[k]:.2f}", ax)
plt.suptitle("the graph's own harmonics: smooth → oscillatory as λ grows", y=1.03)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2968206/3381734998.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [3]:
# ORACLE: on a RING graph, the Laplacian eigenvalues must be the classical DFT frequencies
N = 32
ring = np.zeros((N, N))
for i in range(N): ring[i, (i+1) % N] = ring[i, (i-1) % N] = 1
lam_ring = np.sort(np.linalg.eigvalsh(laplacian(ring)))
lam_theory = np.sort(2 - 2*np.cos(2*np.pi*np.arange(N)/N))   # from the DFT diagonalization
print("max |ring Laplacian eigs − 2−2cos(2πk/N)| =", np.abs(lam_ring - lam_theory).max())
assert np.abs(lam_ring - lam_theory).max() < 1e-9
print("→ classical DSP is the special case: the ring's Fourier basis diagonalizes its Laplacian")

max |ring Laplacian eigs − 2−2cos(2πk/N)| = 3.552713678800501e-15
→ classical DSP is the special case: the ring's Fourier basis diagonalizes its Laplacian


**What just happened.** The ring graph's Laplacian eigenvalues match $2 - 2\cos(2\pi k/N)$ to **3.6e-15** — machine precision. This is the most important cell in the workshop, and it is not a sanity check.

**It is a reduction.** A ring is the graph whose nodes are arranged in a cycle with each connected to its two neighbours — which is exactly the structure of a periodic discrete-time signal. Its Laplacian is diagonalised by the DFT basis, and its eigenvalues are precisely the classical frequency response of the second-difference operator. So **classical DSP is the special case of graph signal processing where the graph is a ring.** The new theory contains the old one rather than competing with it.

That reframing is worth stating in full. Every tool from the DSP track — Fourier analysis, filtering, sampling, convolution — was implicitly assuming a graph all along: a regular, translation-invariant one. Images assume a 2-D grid graph; video adds a temporal edge. GSP simply removes the assumption that the graph is regular, and asks what survives. Most of it does.

**Why $2 - 2\cos(2\pi k/N)$ specifically.** The ring Laplacian applied to a signal computes $2x_n - x_{n+1} - x_{n-1}$, the negative second difference. Feed it the complex exponential $e^{j2\pi kn/N}$ and it comes back scaled by $2 - e^{j2\pi k/N} - e^{-j2\pi k/N} = 2 - 2\cos(2\pi k/N)$. The exponentials are eigenvectors, and those are the eigenvalues — the same "complex exponentials diagonalise shift-invariant operators" fact from [Hilbert Spaces](../Intro_Math/Hilbert_Spaces/Hilbert_Spaces.ipynb), now visibly a graph statement.

Note the shape of that eigenvalue curve: it starts at 0 for $k=0$ and rises to 4 at $k = N/2$. Low $k$ means low $\lambda$ means smooth, exactly as the general theory claims — and $\lambda$ really is playing the role of $\omega$.

**And the general definition earns its keep here.** $x^\top L x = \sum_{(i,j)\in E}(x_i - x_j)^2$ totals the disagreement across edges, so ordering eigenvectors by $\lambda$ orders them from smoothest to most oscillatory. On the sensor network above, eigenvector 0 is constant (every connected graph has exactly one $\lambda = 0$ eigenvector — and the multiplicity of zero counts connected components, a useful diagnostic), while eigenvector 60 alternates sharply between neighbours. That is what high frequency looks like when there is no time axis.

---
### 🕐 Session 2 of 4 — *Filtering on Graphs* (~40 min)
**Goal:** denoise a sensor field with a graph low-pass; make it local with polynomial filters.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (sampling).

---

<details>
<summary>🎓 <b>Teacher notes — Session 2: Filtering on Graphs</b></summary>

**Timing (~40 min).** 8 min filters as functions of $\lambda$ · 10 min the $O(n^3)$ problem · 12 min polynomial filters and locality · 10 min the demo.

**Board first — the translation is one substitution.** A classical filter scales each frequency: $Y(\omega) = H(\omega)X(\omega)$. A graph filter scales each *harmonic*: $y = U h(\Lambda) U^\top x$. Design $h(\lambda)$ exactly as you would design a [frequency response](./Filter_Design.ipynb), with $\lambda$ in place of $\omega$. Say explicitly that nothing new is being invented — the room already knows how to design filters, and the only change is what the basis means.

**Then the practical problem, which motivates everything else.** Computing $U$ requires an eigendecomposition: $O(n^3)$, and $n$ is the number of nodes. A social graph with a million users makes that impossible, and worse, it is *global* — every output node depends on every input node, so it cannot be distributed across a sensor network.

**Polynomial filters solve both problems at once, and the second one is the important one.** $h(L) = \sum_k c_k L^k$ needs only matrix-vector products, so no eigendecomposition. But the deeper property is locality: $L^k x$ at node $i$ depends only on nodes within $k$ hops. Ask the room to verify that for $k=1$ — $(Lx)_i$ involves $i$ and its immediate neighbours — and then argue inductively. **Polynomial order equals filter locality**, and that single sentence is the seed the entire GNN literature grows from. If students take one idea from this session, make it this.

**Ask the room.** "Why does locality matter beyond speed?" Because it makes the filter *implementable in a distributed system*. Each sensor only needs to talk to its neighbours, $k$ times, with no central coordinator and no knowledge of the global graph. That is why polynomial graph filters are what actually gets deployed, and it is why Session 4's GCN layers are stackable.

**Handle the demo's surprise honestly — this needs saying before you run it.** The polynomial approximation scores **8.6 dB** against the exact spectral filter's **7.6 dB**. The approximation *beats* the thing it approximates, which looks impossible. It is not: `h = 1/(1 + 4λ)` was chosen by hand, with `4.0` picked rather than optimised, so it is not the best filter for this problem — it is just *a* reasonable low-pass. The degree-5 least-squares fit does not reproduce it exactly, and on this particular signal and noise realisation the discrepancy happens to help. Do not let the room conclude "polynomials are better"; the honest reading is that the reference filter was not optimal and a 1 dB difference on one realisation is within run-to-run variation anyway.

**Good five-minute extension.** Re-run with several noise seeds and watch the two numbers swap order. That is the cleanest possible demonstration that a single-run difference of this size means nothing — and it is a habit worth installing generally.
</details>

## 3. Graph Filters

💡 **Intuition.** A graph filter scales each harmonic: $y = U h(\Lambda) U^T x$ — design $h(\lambda)$ exactly like a [filter response](./Filter_Design.ipynb), with $\lambda$ replacing $\omega$. The practical twist: eigendecomposition is $O(n^3)$, but a **polynomial** filter $h(L) = \sum_k c_k L^k$ needs only matrix-vector products — and $L^k x$ touches only $k$-hop neighbors, so polynomial order = *filter locality*. That locality is the seed GNNs grow from.

In [4]:
# denoise a smooth temperature field over the sensor network
x_clean = U[:, :4] @ (rng.standard_normal(4) * [3, 2, 1.5, 1])   # smooth by construction
x_noisy = x_clean + 0.6 * rng.standard_normal(n_nodes)

h = 1.0 / (1.0 + 4.0 * lam)                                  # graph low-pass (Tikhonov)
x_filt = U @ (h * (U.T @ x_noisy))

# local polynomial approximation of the same filter (Chebyshev-lite: least-squares fit)
Vand = np.vander(lam, 6, increasing=True)
c, *_ = np.linalg.lstsq(Vand, h, rcond=None)
x_poly = np.zeros(n_nodes); Lk_x = x_noisy.copy()
for k in range(6):
    x_poly += c[k] * Lk_x
    Lk_x = L @ Lk_x

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (sig, t) in zip(axes, [(x_noisy, "noisy sensors"), (x_filt, "spectral low-pass"),
                                (x_poly, "5-hop polynomial (no eig needed)")]):
    draw(sig, t, ax)
plt.tight_layout(); plt.show()
for name, xh in [("spectral", x_filt), ("polynomial", x_poly)]:
    print(f"{name:10s} SNR gain: {10*np.log10(np.var(x_noisy-x_clean)/np.var(xh-x_clean)):.1f} dB")

spectral   SNR gain: 7.6 dB
polynomial SNR gain: 8.6 dB


/tmp/ipykernel_2968206/2748018384.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**What just happened.** Both filters cleaned up the noisy sensor field — **7.6 dB** for the exact spectral filter, **8.6 dB** for the 5-hop polynomial. Denoising on an irregular network, using a frequency axis that only exists because we defined one.

**The design was a one-line translation.** `h = 1.0 / (1.0 + 4.0 * lam)` is a low-pass: near 1 at $\lambda \approx 0$ (smooth harmonics pass), rolling off as $\lambda$ grows (oscillatory harmonics suppressed). That is exactly how you would design a [classical frequency response](./Filter_Design.ipynb), with $\lambda$ substituted for $\omega$. And it works for the same reason as always: the signal was built from the first four harmonics, so it is smooth, while the added noise spreads across *all* harmonics. Low-pass keeps the signal and discards most of the noise.

**But look at the polynomial version, because it is the practically important one.** Computing $U$ requires an $O(n^3)$ eigendecomposition, which is hopeless for a graph with a million nodes and — worse — *global*, since every output depends on every input. The polynomial filter $\sum_k c_k L^k$ needs only matrix–vector products, and it has a much deeper property: **$L^k x$ at node $i$ depends only on nodes within $k$ hops.** Polynomial order *is* filter locality.

That is what makes graph filtering deployable. A 5-hop filter can run on the sensor network itself — each node exchanging values with its immediate neighbours, five times, with no central computer and no node ever knowing the global graph. It is also the seed from which every GNN grows, which Session 4 makes explicit.

**Now the result that looks impossible: the approximation beat the exact filter.** 8.6 dB against 7.6 dB — the polynomial fit outperformed the thing it was fitting. Resolve it carefully rather than moving on.

The spectral filter is *exact* for the response `h`, but `h` itself was chosen by hand: the `4.0` was picked, not optimised. So it is a reasonable low-pass, not the optimal one for this problem, and there is no theorem saying it should win. The degree-5 least-squares fit does not reproduce `h` exactly — its mismatch is presumably a slightly more aggressive roll-off — and on this particular signal and noise realisation that mismatch happens to help.

The honest reading is therefore **not** "polynomials are better." It is that the reference filter was never optimal, and a 1 dB gap measured on a single noise realisation is within run-to-run variation regardless. Re-run with a few different seeds and the ordering will swap. A one-run difference of this size is not a finding, and treating it as one is the exact mistake this workshop's oracle-driven style exists to prevent.

---
### 🕐 Session 3 of 4 — *Sampling on Graphs* (~35 min)
**Goal:** which sensors can you afford to lose? Bandlimited recovery from a subset of nodes.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (GNNs).

---

<details>
<summary>🎓 <b>Teacher notes — Session 3: Sampling on Graphs</b></summary>

**Timing (~35 min).** 8 min what bandlimited means on a graph · 10 min the recovery argument · 10 min the demo including the failure · 7 min where-to-sample.

**Board first — translate Nyquist.** Classical sampling says a bandlimited signal is determined by samples at twice its highest frequency. The graph version: a signal in the span of the first $K$ harmonics has only $K$ unknown coefficients, so $K$ well-chosen node readings determine all $n$ values. Counting unknowns is the whole argument, and it is worth doing explicitly — students expect something deeper and the simplicity is the point.

**Frame it as a subspace problem, which is what it is.** $x = U_{:,:K}\,c$ with $c$ unknown. Read $m$ nodes and you have $m$ equations in $K$ unknowns. If $m \geq K$ and the selected rows of $U_{:,:K}$ are well-conditioned, `lstsq` recovers $c$ exactly and then reconstructs every node. This is the subspace version of the [Compressed Sensing](./Compressed_Sensing.ipynb) logic — there the support was unknown and the problem was combinatorial; here the subspace is *known*, so it is a linear solve.

**Make sure the failure case is understood as aliasing.** Three sensors for four unknowns is underdetermined — infinitely many bandlimited signals fit, and `lstsq` returns the minimum-norm one, which is wrong. Say plainly this is **aliasing**, graph edition: below the required sample count, distinct signals become indistinguishable, and no algorithm resolves it because the information is not there. The 0.13 error is not a numerical issue.

**Then the part with real engineering content: *which* nodes.** "Well-chosen" is doing serious work. The relevant object is the conditioning of $U_{\text{sel},:K}$ — you need rows that make the $K$ harmonics distinguishable. Ask what happens if all 12 sensors are clustered in one corner of the network: they see nearly the same values, the rows are nearly parallel, the system is ill-conditioned, and recovery is numerically hopeless even though $m > K$. This is exactly the array-geometry lesson from [Array Processing](./Array_Processing.ipynb) — sensor *placement* matters as much as sensor *count*.

**Ask the room.** "You have budget for 12 sensors on a 500-node water network. How do you place them?" There is a real literature on this (greedy determinant maximisation, leverage-score sampling, E-optimal design), and the honest answer is that it is an optimisation problem, not a formula. Students find it satisfying that "where should the sensors go" is a well-posed mathematical question with a computable answer.

**Note the demo's convenience.** `x_band = x_clean` is bandlimited *by construction* — it was built from exactly the first four harmonics. Real signals are only approximately bandlimited, so recovery has an error floor set by the out-of-band energy rather than by the solve. That is why the max error here is 2e-15 and would never be that on real data; say so, or the result looks better than the method is.
</details>

## 4. Nyquist for Networks

💡 **Intuition.** If a graph signal is **bandlimited** — lives in the span of the first $K$ harmonics — then $K$ well-chosen node readings determine *all* $n$: solve the little least-squares system in the known coefficients ([Compressed Sensing's](./Compressed_Sensing.ipynb) logic, subspace version). 'Well-chosen' matters exactly like array geometry: sample nodes that make the harmonics distinguishable, not clustered clones of each other.

In [5]:
K, m = 4, 12
x_band = x_clean                                             # bandlimited by construction (K=4)
sel = rng.choice(n_nodes, m, replace=False)                  # random sensor subset
coef, *_ = np.linalg.lstsq(U[sel, :K], x_band[sel], rcond=None)
x_rec = U[:, :K] @ coef
print(f"recover all {n_nodes} nodes from {m} sensors: max error {np.abs(x_rec - x_band).max():.2e}")
assert np.abs(x_rec - x_band).max() < 1e-8

# and the failure mode: measure fewer than K nodes → underdetermined
sel_bad = sel[:3]
coef_bad, *_ = np.linalg.lstsq(U[sel_bad, :K], x_band[sel_bad], rcond=None)
print(f"with only 3 < K sensors: max error {np.abs(U[:, :K] @ coef_bad - x_band).max():.2f}  (aliasing, graph edition)")

recover all 80 nodes from 12 sensors: max error 2.22e-15
with only 3 < K sensors: max error 0.13  (aliasing, graph edition)


**What just happened.** All **80** node values reconstructed from **12** sensor readings, to a maximum error of **2.2e-15** — machine precision. And then the failure: drop to 3 sensors, below $K = 4$, and the error jumps to **0.13**.

**The argument is just counting.** A bandlimited graph signal lives in the span of the first $K$ harmonics, so it has only $K$ unknown coefficients no matter how many nodes the graph has. Read $m \geq K$ nodes and you have $m$ equations in $K$ unknowns; `lstsq` solves for the coefficients and $U_{:,:K}c$ reconstructs everything. Students often expect a deeper mechanism — there isn't one, and that is the point. This is the subspace version of the [Compressed Sensing](./Compressed_Sensing.ipynb) logic, but *easier*: there the support was unknown and the search combinatorial; here the subspace is known and it is a linear solve.

**The 3-sensor failure is aliasing, graph edition.** Three equations cannot determine four unknowns. Infinitely many bandlimited signals pass through those three readings, `lstsq` returns the minimum-norm one, and it is wrong. That is not a numerical problem and no better solver fixes it — below the required sample count, distinct signals are genuinely indistinguishable, exactly as sub-Nyquist sampling makes distinct sinusoids indistinguishable in classical DSP. The information is not there.

**Now the phrase carrying the real engineering: "well-chosen".** The recovery needs the selected rows of $U_{:,:K}$ to be well-conditioned. Twelve sensors clustered in one corner of the network would see nearly identical values, making those rows nearly parallel — the system would be numerically hopeless even with $m \gg K$. **Sensor placement matters as much as sensor count**, which is precisely the array-geometry lesson from [Array Processing](./Array_Processing.ipynb) transplanted onto an irregular graph.

That turns "where should the sensors go?" into a well-posed optimisation: choose the node subset maximising the conditioning (or the determinant, or the smallest singular value) of $U_{\text{sel},:K}$. There is a real literature here — greedy determinant maximisation, leverage-score sampling, E-optimal design — and it is genuinely useful. If you have budget for 12 sensors on a 500-node water network, this is the question you are asking.

**One honest caveat about how easy this instance is.** `x_band = x_clean` was *constructed* from exactly the first four harmonics, so it is perfectly bandlimited and the residual is pure floating-point. Real signals are only approximately bandlimited: their out-of-band energy sets an error floor that no sampling scheme removes, so you would never see 2e-15 on measured data. The mechanism is real; the precision here is a property of the synthetic setup.

---
### 🕐 Session 4 of 4 — *Message Passing = Learned Graph Filters* (~40 min)
**Goal:** build a GCN from scratch; classify nodes; see it as Session 2 with trained coefficients.
**Builds on:** Session 3; [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb).

---

<details>
<summary>🎓 <b>Teacher notes — Session 4: Message Passing = Learned Graph Filters</b></summary>

**Timing (~40 min).** 10 min the GCN layer decomposed · 8 min why it is Session 2 with learned coefficients · 12 min the demo and the ablation · 10 min oversmoothing and the honest reading.

**Board first — take the layer apart into three named steps.** Aggregate ($\hat A X$: average your neighbours — a fixed 1-hop low-pass), transform (a learned linear map), nonlinearity. That is the entire graph-convolution layer. Then the punchline: stacking $k$ layers gives a $k$-hop receptive field, which is **exactly Session 2's polynomial filter of order $k$**, with coefficients chosen by gradient descent instead of by least squares. GNNs are not a new object; they are learned graph filters, and a room that has just done Session 2 can see that directly.

**Make the CNN connection explicit.** A CNN shares weights across spatial positions and looks at local neighbourhoods; a GCN shares weights across *nodes* and looks at graph neighbourhoods. Given Session 1's ring oracle — classical DSP is GSP on a regular graph — a CNN is literally a GCN on a grid. Ask the room what makes the graph case harder: neighbourhoods vary in size and have no canonical ordering, which is why you average (a permutation-invariant operation) rather than apply an ordered filter mask.

**Set up the ablation properly before running it.** Same architecture, same features, same 10 labels, same optimiser — the *only* difference is whether $\hat A$ appears. That makes it a controlled experiment rather than a demonstration, and worth naming as such.

**Then handle the two numbers carefully, because one of them is strange.** GCN 88.6%, MLP **41.4%**. With two roughly balanced classes, 41.4% is *below chance* — the MLP is not merely uninformative, it is actively wrong. Ask the room what that means. With 10 labelled examples and one deliberately useless feature, the MLP overfits noise in the training set and generalises anti-correlated with the truth; on a small test set this is entirely ordinary variance. **Do not present 41.4% as "the MLP is 47 points worse"** — the meaningful comparison is 88.6% against the ~50% a coin gets, and the MLP landing slightly below chance is noise, not signal.

**Why the GCN wins is the real content.** Ten labels classify eighty nodes because the *graph* carries information the features do not. Neighbouring nodes tend to share labels (homophily), so averaging over neighbours propagates the ten labels outward and denoises the useful feature. Say explicitly that this only works when homophily holds — on a graph where neighbours tend to *differ*, the same averaging destroys the signal, and there is a real literature on heterophilous GNNs for exactly that case.

**Close on oversmoothing, since the notebook flags it.** Stack many layers and every node's representation converges toward the same value — repeated low-pass filtering drives everything to the $\lambda \approx 0$ eigenvector, which is constant. That is the graph version of over-aggressive smoothing, and it is why most GCNs are 2–3 layers deep while CNNs are 50+. Framed through Session 2, it is obvious rather than mysterious: you are applying the same low-pass over and over.
</details>

## 5. GNNs, Demystified

💡 **Intuition.** A graph-convolution layer is: *average your neighbors (a fixed 1-hop low-pass $\hat{A} = \tilde{D}^{-1/2}\tilde{A}\tilde{D}^{-1/2}$), then apply a learned linear map and a nonlinearity*. Stack $k$ layers ⇒ $k$-hop receptive field — precisely Session 2's polynomial filters with coefficients chosen by gradient descent. It's the [CNN story](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) (weight sharing + locality) generalized to irregular neighborhoods.

In [6]:
import torch
import torch.nn as nn
torch.manual_seed(0)

# two-community node classification (a planted partition on top of geometry)
comm = (pos[:, 0] + 0.25*rng.standard_normal(n_nodes) > 0.5).astype(int)
A2 = A.copy()
for i in range(n_nodes):                                   # densify within communities
    same = np.where((comm == comm[i]) & (np.arange(n_nodes) != i))[0]
    for j in rng.choice(same, 2): A2[i, j] = A2[j, i] = 1
At = A2 + np.eye(n_nodes)
Dh = np.diag(1/np.sqrt(At.sum(1)))
Ahat = torch.tensor(Dh @ At @ Dh, dtype=torch.float32)

feats = torch.tensor(np.stack([rng.standard_normal(n_nodes),  # useless feature
                               comm + 1.2*rng.standard_normal(n_nodes)], 1),  # noisy hint
                     dtype=torch.float32)
labels = torch.tensor(comm)
train_mask = torch.zeros(n_nodes, dtype=bool); train_mask[rng.choice(n_nodes, 10, replace=False)] = True

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.W1, self.W2 = nn.Linear(2, 16), nn.Linear(16, 2)
    def forward(self, X):
        H = torch.relu(self.W1(Ahat @ X))       # aggregate neighbors → transform → nonlinearity
        return self.W2(Ahat @ H)

class MLP(nn.Module):                            # ablation: same net, NO graph
    def __init__(self):
        super().__init__()
        self.W1, self.W2 = nn.Linear(2, 16), nn.Linear(16, 2)
    def forward(self, X): return self.W2(torch.relu(self.W1(X)))

for name, model in [("GCN (uses edges)", GCN()), ("MLP (ignores edges)", MLP())]:
    opt = torch.optim.Adam(model.parameters(), lr=0.02)
    for step in range(300):
        loss = nn.functional.cross_entropy(model(feats)[train_mask], labels[train_mask])
        opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        acc = (model(feats).argmax(1) == labels)[~train_mask].float().mean()
    print(f"{name:20s}: {acc:.1%} accuracy on unseen nodes (10 labeled examples!)")

GCN (uses edges)    : 88.6% accuracy on unseen nodes (10 labeled examples!)
MLP (ignores edges) : 41.4% accuracy on unseen nodes (10 labeled examples!)


**What just happened.** Ten labelled nodes, eighty to classify. The GCN reaches **88.6%** on the unseen nodes; the identical network without the graph reaches **41.4%**.

**This is a controlled ablation, which is what makes it worth anything.** Same architecture, same two input features, same ten labels, same optimiser, same number of steps. The *only* difference is whether $\hat A$ appears in the forward pass. So the gap is attributable to the graph and to nothing else.

**But read the 41.4% correctly.** With two roughly balanced classes, chance is about 50% — so the MLP scored *below* chance. It is not merely uninformative, it is anti-correlated with the truth. That is not evidence of the graph being 47 points valuable; it is what overfitting ten examples looks like. One of the two input features is deliberately useless random noise, and with ten training points the MLP can fit that noise and generalise backwards. On a 70-node test set, landing a few points either side of chance is ordinary variance.

**The honest comparison is 88.6% against ~50%**, and the MLP's exact value below chance is noise rather than a measurement. Anyone quoting "GCN beats MLP by 47 points" from this cell is over-reading it.

**Why the graph is worth so much here.** The features barely distinguish the classes — one is pure noise, the other a noisy hint. What the GCN adds is *structure*: neighbouring nodes tend to share labels (homophily), so averaging over neighbours both propagates the ten known labels outward and denoises the weak feature. Ten labels classify eighty nodes because the edges carry information that the node features do not.

That condition is load-bearing and worth naming: it only works under **homophily**. On a graph where connected nodes tend to *differ* — a fraud network where fraudsters transact with non-fraudsters, say — this same averaging destroys the signal, and specialised heterophilous architectures exist for exactly that case.

**And the layer is Session 2 with learned coefficients.** Decompose the forward pass: `Ahat @ X` averages neighbours (a fixed 1-hop low-pass), `W1` applies a learned linear map, `relu` adds the nonlinearity. Stack two layers and the receptive field is 2 hops — which is precisely a **degree-2 polynomial graph filter**, with the coefficients found by gradient descent instead of by least squares. GNNs are not a new mathematical object; they are the polynomial filters you built an hour ago, trained rather than designed.

Seen that way, the [CNN](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) connection is immediate too. A CNN shares weights across spatial positions with local receptive fields; a GCN shares weights across nodes with graph-local receptive fields. Given Session 1's ring oracle, a CNN is a GCN whose graph happens to be a grid — the harder part on a general graph is only that neighbourhoods vary in size and have no canonical ordering, which is why you *average* (permutation-invariant) rather than apply an ordered mask.

**One consequence worth knowing.** Repeated neighbour-averaging is repeated low-pass filtering, and enough of it drives every node toward the $\lambda \approx 0$ eigenvector — which is constant. That is **oversmoothing**: stack too many layers and all representations converge to the same value. It is why most GCNs are 2–3 layers deep while CNNs run to 50+, and through Session 2's lens it is obvious rather than mysterious.

Ten labels classify eighty nodes because the graph *propagates* them — message passing is label smoothing through a learned low-pass. (Also visible here: stack too many layers and everything averages toward mush — *oversmoothing*, the graph version of over-aggressive low-pass filtering.)

## 6. Conclusion

The Laplacian gives every network a Fourier basis (reducing to the DFT on a ring — verified); filters are functions of $L$, made local by polynomials; bandlimited signals need only $K$ good sensors; and GNNs are those polynomial filters with learned coefficients. Classical DSP was the special case all along.

---
## Where next

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) — the eigen-machinery, if it felt fast.
- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — the regular-grid special case.
- [Statistical SP](./Statistical_Signal_Processing.ipynb) — stochastic graph signals are an open research door.